# Chapter 11 &mdash; Closure Properties of Context-Free Languages

**Concept 14 of the Chapter 11 decomposition:** *Closure Properties of Context-Free Languages*

Closed under union, concatenation, star and reversal; <b>not</b> under intersection or complement.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-CFL-Closure-Properties/Concept-CFL-Closure-Properties.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Context-free languages are closed under:

* **union** &mdash; $S \to S_1 \mid S_2$;
* **concatenation** &mdash; $S \to S_1 S_2$;
* **star** &mdash; $S \to S_1 S \mid \varepsilon$;
* **reversal** &mdash; reverse every right-hand side.

Each is a one-line grammar construction, which is the whole proof.

They are **not** closed under **intersection** or **complement**. The witness:
$\{a^nb^nc^m\}$ and $\{a^mb^nc^n\}$ are both context-free, but their intersection is
$\{a^nb^nc^n\}$, which is not (Concept 19).

That is a sharp break from the regular languages, which are closed under everything &mdash;
and it is why a product construction for PDA cannot exist (Chapter 12).

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The four constructions

In [ ]:
# Every symbol in this toolkit is ONE character, so renaming has to draw
# fresh single letters from a pool rather than append a digit.
POOL = 'ZYXWVUTRQPONMLKJIHGFEDCBA'

def fresh(avoid, n):
    out = [ch for ch in POOL if ch not in avoid][:n]
    assert len(out) == n, "ran out of fresh nonterminal letters"
    return out

def relabel(G, avoid):
    news = fresh(set(avoid) | set(G['Sigma']), len(G['N']))
    m = dict(zip(sorted(G['N']), news))
    P = {m[A]: [tuple(m.get(x, x) for x in r) for r in G['P'][A]] for A in G['P']}
    return dict(N=set(m.values()), Sigma=set(G['Sigma']), S=m[G['S']], P=P)

def combine(G1, G2, rhs):
    A = relabel(G1, set())
    B = relabel(G2, A['N'])
    top = fresh(A['N'] | B['N'] | A['Sigma'] | B['Sigma'], 1)[0]
    P = dict(A['P']); P.update(B['P'])
    P[top] = [tuple(A['S'] if ch == 'X' else B['S'] if ch == 'Y' else ch
                    for ch in r) for r in rhs]
    return dict(N=A['N'] | B['N'] | {top}, Sigma=A['Sigma'] | B['Sigma'],
                S=top, P=P)

def union(G1, G2):  return combine(G1, G2, ['X', 'Y'])
def concat(G1, G2): return combine(G1, G2, ['XY'])
def star(G1):
    A = relabel(G1, set())
    top = fresh(A['N'] | A['Sigma'], 1)[0]
    P = dict(A['P']); P[top] = [(), (A['S'], top)]
    return dict(N=A['N'] | {top}, Sigma=set(A['Sigma']), S=top, P=P)
def reverse(G):
    return dict(N=set(G['N']), Sigma=set(G['Sigma']), S=G['S'],
                P={A: [tuple(reversed(r)) for r in v] for A, v in G['P'].items()})

### Two context-free languages to combine

In [ ]:
AnBn = mkg({'S': ["", "aSb"]})
Dyck = mkg({'S': ["", "cSd", "SS"]})

## 3. Tests

**Union** and **concatenation**.

In [ ]:
U = union(AnBn, Dyck)
print("union, up to length 4 :", language(U, 4))
assert set(language(AnBn, 4)) <= set(language(U, 4))
assert set(language(Dyck, 4)) <= set(language(U, 4))

C = concat(AnBn, Dyck)
print("concat, up to length 4 :", language(C, 4))
assert 'abcd' in language(C, 4)

**Star**.

In [ ]:
St = star(AnBn)
L = language(St, 6)
print("star, up to length 6 :", L)
assert '' in L and 'abab' in L and 'aabb' in L

**Reversal**.

In [ ]:
R = reverse(AnBn)
print("reverse of a^n b^n :", language(R, 6))
assert set(language(R, 6)) == {w[::-1] for w in language(AnBn, 6)}

**Not** closed under intersection: the classic witness.

In [ ]:
def in1(s):   # a^n b^n c^m
    i = len(s) - len(s.lstrip('a')); rest = s[i:]
    j = len(rest) - len(rest.lstrip('b')); k = len(rest) - j
    return s == 'a'*i + 'b'*j + 'c'*k and i == j
def in2(s):   # a^m b^n c^n
    i = len(s) - len(s.lstrip('a')); rest = s[i:]
    j = len(rest) - len(rest.lstrip('b')); k = len(rest) - j
    return s == 'a'*i + 'b'*j + 'c'*k and j == k

G1 = mkg({'S': ["XC"], 'X': ["", "aXb"], 'C': ["", "cC"]})
G2 = mkg({'S': ["AY"], 'A': ["", "aA"], 'Y': ["", "bYc"]})
L1, L2 = set(language(G1, 6)), set(language(G2, 6))
assert all(in1(w) for w in L1) and all(in2(w) for w in L2)
inter = sorted(L1 & L2, key=lambda s: (len(s), s))
print("both are context-free; their intersection starts :", inter)
assert inter[:3] == ['', 'abc', 'aabbcc']
print("\n... which is a^n b^n c^n -- NOT context-free (Concept 19).")

**Not** closed under complement either &mdash; it follows from the above.

In [ ]:
print("If CFLs were closed under complement, then since they ARE closed under")
print("union, DeMorgan would give closure under intersection:")
print("   L1 & L2 = complement( complement(L1) + complement(L2) )")
print("Intersection fails, so complement must fail too.")
print()
print("Contrast Chapter 6: regular languages are closed under ALL of these.")

## 4. Exercises


1. Write out the star construction and argue it is correct.
2. Is the class of CFLs closed under **homomorphism**? Under **inverse** homomorphism?
3. Why does the DeMorgan argument need closure under union?

In [ ]:
# Your work for the exercises above.